# Kestrel Milestone 1: Raw Data Profiling

This notebook profiles the original nine M1 inputs before cleaning. It deliberately excludes `activity_part1.parquet` through `activity_part6.parquet` and does not use the M2-only stream or incident-label files. Run from the repository root or from `M1/`.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import sys
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if ROOT.name == 'M1':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
M1 = ROOT / 'M1'
print('Source root:', ROOT)
print('Selected Parquet:', ROOT / 'agent_activity.parquet')

## Activity Parquet: Before Cleaning

In [ ]:
parquet_path = ROOT / 'agent_activity.parquet'
parquet_file = pq.ParquetFile(parquet_path)
activity = parquet_file.read().to_pandas()
print('Parquet schema:\n', parquet_file.schema_arrow)
print('Rows:', parquet_file.metadata.num_rows, 'row groups:', parquet_file.num_row_groups)
print('Full-row duplicates:', activity.duplicated().sum())
print('Duplicate event_id rows:', activity['event_id'].duplicated().sum())
print('Null rates:\n', activity.isna().mean().sort_values(ascending=False).to_string())
print('Cardinalities:\n', activity.nunique(dropna=False).to_string())
print('Event types:\n', activity['event_type'].value_counts(dropna=False).to_string())
print('Statuses:\n', activity['status'].value_counts(dropna=False).to_string())
for column in ['event_time', 'ingest_time', 'finished_at']:
    raw = activity[column].dropna().astype(str)
    epoch_ms = raw.str.fullmatch(r'[0-9]{13}').sum()
    explicit_zone = raw.str.contains(r'(?:Z|[+-][0-9]{2}:?[0-9]{2})$', regex=True).sum()
    parsed = pd.to_datetime(raw, format='mixed', utc=True, errors='coerce', dayfirst=True)
    print(f'{column}: present={len(raw):,}; epoch_ms={epoch_ms:,}; explicit_zone={explicit_zone:,}; parse_fail={parsed.isna().sum():,}; min={parsed.min()}; max={parsed.max()}')

## Reference CSVs and Model Pricing: Before Cleaning

In [ ]:
def profile_csv(filename, natural_key=None):
    frame = pd.read_csv(ROOT / filename)
    print(f'\n{filename}: rows={len(frame):,}, columns={len(frame.columns)}')
    print('null rates:\n', frame.isna().mean().to_string())
    print('cardinalities:\n', frame.nunique(dropna=False).to_string())
    print('exact duplicate rows:', frame.duplicated().sum())
    if natural_key:
        print('natural-key duplicate rows:', frame.duplicated(natural_key).sum(), 'key:', natural_key)
    return frame

tenants = profile_csv('tenants.csv', ['tenant_id'])
agents = profile_csv('agents.csv', ['agent_id', 'release_version'])
tools = profile_csv('tools.csv', ['tool_id', 'tool_version'])
pricing = profile_csv('model_pricing.csv', ['model_id', 'effective_from'])
print('Agent-version frequency:\n', agents.groupby('agent_id')['release_version'].agg(list).to_string())
print('Tool identifiers and versions:\n', tools[['tool_id', 'tool_version', 'tool_name']].to_string(index=False))
print('Model price rows:\n', pricing.to_string(index=False))

## Nested Sandbox Policy JSON: Before Cleaning

In [ ]:
policy_objects = json.loads((ROOT / 'sandbox_policies.json').read_text(encoding='utf-8'))
policies_flat = pd.json_normalize(policy_objects, sep='.')
print('Policy versions:', len(policy_objects))
print('Flattened columns:', policies_flat.columns.tolist())
print('Null rates:\n', policies_flat.isna().mean().to_string())
print('Profile/version keys:\n', policies_flat[['sandbox_profile_id', 'policy_version', 'valid_from', 'valid_to']].to_string(index=False))
print('Policies missing network block:', sum('network' not in policy for policy in policy_objects))

## Network Flows CSV: Before Cleaning

In [ ]:
flows = profile_csv('network_flows.csv', ['flow_id'])
flow_time = pd.to_datetime(flows['flow_time'], format='mixed', utc=True, errors='coerce')
print('Flow timestamp parse failures:', flow_time.isna().sum())
print('Flow timestamp range:', flow_time.min(), 'to', flow_time.max())
print('Verdicts:\n', flows['verdict'].value_counts(dropna=False).to_string())
print('Protocols:\n', flows['protocol'].value_counts(dropna=False).to_string())
print('Destination ports:\n', flows['dst_port'].value_counts(dropna=False).to_string())

## Nested SIEM Alert JSON: Before Cleaning

In [ ]:
siem_doc = json.loads((ROOT / 'siem_alerts.json').read_text(encoding='utf-8'))
alerts = pd.json_normalize(siem_doc['alerts'], sep='.')
print('Envelope fields:', {key: value for key, value in siem_doc.items() if key != 'alerts'})
print('Alerts:', len(alerts))
print('Null rates:\n', alerts.isna().mean().to_string())
print('Alert ID duplicates:', alerts['alert_id'].duplicated().sum())
print('Source system counts:\n', alerts['source_system'].value_counts().to_string())
print('Severity counts:\n', alerts['severity'].value_counts().to_string())
alert_time = pd.to_datetime(alerts['detected_at'], format='mixed', utc=True, errors='coerce')
print('Detected-at parse failures:', alert_time.isna().sum(), 'range:', alert_time.min(), 'to', alert_time.max())
print('Observed timezone offsets:\n', alerts['detected_at'].astype(str).str.extract(r'([+-][0-9]{2}:[0-9]{2})$')[0].value_counts().to_string())
print('Alerts with source IP present among flow IPs:', alerts['source_ip'].isin(flows['src_ip']).sum(), '/', len(alerts))

## Sandbox Audit Syslog: Before Cleaning

In [ ]:
audit_pattern = re.compile(r'^(?P<mon>[A-Z][a-z]{2})\s+(?P<day>\d{1,2})\s+(?P<hms>\d{2}:\d{2}:\d{2})\s+(?P<host>\S+)\s+(?P<process>[^[]+)\[(?P<pid>\d+)\]:\s+session=(?P<session>\S+)\s+instance=(?P<instance>\S+)\s+decision=(?P<decision>\S+)\s+action=(?P<action>\S+)\s+target=(?P<target>\S+)\s+reason=(?P<reason>\S+)\s+policy_version=(?P<policy>\S+)\s*$')
audit_lines = (ROOT / 'sandbox_audit.log').read_text(encoding='utf-8', errors='replace').splitlines()
audit_matches = [audit_pattern.match(line) for line in audit_lines]
parsed_audit = [match.groupdict() for match in audit_matches if match]
print('Physical lines:', len(audit_lines), 'regex failures:', sum(match is None for match in audit_matches))
print('Parsed hosts:', len({row['host'] for row in parsed_audit}))
print('Decision counts:\n', pd.Series([row['decision'] for row in parsed_audit]).value_counts(dropna=False).to_string())
print('Action counts:\n', pd.Series([row['action'] for row in parsed_audit]).value_counts(dropna=False).to_string())

## Findings Before Cleaning

The cells above are the evidence for row counts, schemas, nulls, cardinalities, duplicate keys, timestamp forms, and distributions. The observed defects include duplicate activity event IDs, unrecognized event types, unknown activity tenant/agent identifiers, multiple event-time encodings including malformed values, negative numeric values, invalid retry values, alias tool names, a duplicate tool catalogue row, unresolved tenant display-name similarity, incomplete pricing for the nano model, nullable flow destination metadata, malformed audit decisions, and SIEM alerts that do not all correspond to platform flows.

Cleaning decisions follow the project description: deduplicate by declared source key and keep earliest activity ingest time; resolve release-version prefix and documented tool-name aliases only; reject invalid/unparseable source records with one closed-enum reason; preserve optional missing values as null; infer the sandbox host-local offset from cross-source time alignment; apply policy validity windows at event time; and leave multi-candidate or absent SIEM-flow matches unlinked. No M2 labels or hold-out stream rows are used.

In [ ]:
import csv

canonical_path = M1 / 'data' / 'canonical_activity.csv'
rejects_path = M1 / 'data' / 'fact_rejected_record.csv'
metrics = json.loads((M1 / 'data' / 'metrics.json').read_text(encoding='utf-8'))
correlation = json.loads((M1 / 'data' / 'correlation_summary.json').read_text(encoding='utf-8'))

def csv_data_rows(path):
    with path.open(encoding='utf-8', newline='') as source:
        return max(sum(1 for _ in csv.reader(source)) - 1, 0)

print('Canonical rows:', csv_data_rows(canonical_path))
print('Rejected rows:', csv_data_rows(rejects_path))
print('Correlation counts:', correlation)
print('Metrics:', json.dumps(metrics, indent=2, default=str))